# WARAS — fitting parameter squat (Chen et al. 2026)
Ini bukan training deep learning. Notebook ini cuma mencari angka untuk aturan di `lib/squat.ts`:
- `k, b`: regresi linear `sudut_panggul = k * sudut_lutut + b` dari squat standar
- `tol`: `min(3*sigma, 15)` derajat
- `delta`: toleransi kedalaman (panggul boleh di atas lutut sejauh `delta * panjang_paha`)

Output: `waras_params.json` → taruh di `lib/waras_params.json` di repo, push, Vercel deploy ulang otomatis.

**Langkah:** Runtime → Run all. Di sel upload, pilih file data dari Zenodo (link ada di bagian *Data Availability* artikel, doi:10.3390/computers15050293).

In [ ]:
import numpy as np, pandas as pd, json, glob, os, zipfile
from google.colab import files
SEED = 42

## 1. Upload data Zenodo (CSV atau ZIP berisi CSV)

In [ ]:
up = files.upload()
os.makedirs('data', exist_ok=True)
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall(f'data/{name[:-4]}')  # keep zip name in path (label hint)
    else:
        os.replace(name, f'data/{name}')
paths = sorted(glob.glob('data/**/*.csv', recursive=True))
print(len(paths), 'file CSV'); print(*paths[:20], sep='\n')
pd.read_csv(paths[0]).head()

## 2. Sesuaikan dengan format data (SEKALI SAJA)
Lihat output sel di atas, lalu sesuaikan `load_clip` supaya tiap CSV jadi DataFrame per-frame berisi kolom:
`subject, clip, label (1=standar, 0=tidak standar), sx, sy, hx, hy, kx, ky, ax, ay`
(koordinat bahu, panggul, lutut, pergelangan kaki; satu sisi badan, yang menghadap kamera).

Kalau datanya sudah berisi sudut panggul & lutut, isi juga kolom `hip` dan `knee`; kalau tidak, dihitung otomatis.
`ASPECT` = lebar/tinggi video kalau koordinat dinormalisasi 0–1 (MediaPipe); isi 1 kalau koordinat dalam piksel.

In [ ]:
ASPECT = 1.0

def load_clip(path):
    d = pd.read_csv(path)
    # >>> CONTOH: ganti nama kolom sesuai data Zenodo <<<
    # MediaPipe index: bahu 11/12, panggul 23/24, lutut 25/26, pergelangan kaki 27/28
    out = pd.DataFrame({
        'sx': d['x_11'], 'sy': d['y_11'],
        'hx': d['x_23'], 'hy': d['y_23'],
        'kx': d['x_25'], 'ky': d['y_25'],
        'ax': d['x_27'], 'ay': d['y_27'],
    })
    out['clip'] = os.path.basename(path)
    out['subject'] = os.path.basename(path).split('_')[0]      # mis. "S01_standard_03.csv"
    out['label'] = 0 if any(w in path.lower() for w in ['shallow', 'good', 'non']) else 1
    return out

df = pd.concat([load_clip(p) for p in paths], ignore_index=True)
print(df.groupby('label').clip.nunique().rename({1: 'standar', 0: 'tidak standar'}))
df.head()

## 3. Sudut sendi (hukum cosinus), sama persis dengan `angle()` di `lib/squat.ts`

In [ ]:
def angle(ax_, ay_, bx, by, cx, cy):
    ab = np.hypot(ax_ - bx, ay_ - by); cb = np.hypot(cx - bx, cy - by); ac = np.hypot(ax_ - cx, ay_ - cy)
    cos = np.clip((ab**2 + cb**2 - ac**2) / (2 * ab * cb), -1, 1)
    return np.degrees(np.arccos(cos))

X = lambda c: df[c] * ASPECT
if 'hip' not in df:  df['hip']  = angle(X('sx'), df.sy, X('hx'), df.hy, X('kx'), df.ky)
if 'knee' not in df: df['knee'] = angle(X('hx'), df.hy, X('kx'), df.ky, X('ax'), df.ay)
df['thigh'] = np.hypot(X('hx') - X('kx'), df.hy - df.ky)
df = df.dropna(subset=['hip', 'knee', 'thigh'])
df[['hip', 'knee', 'thigh']].describe()

## 4. Split 80/20 per subjek (bukan per frame, supaya tidak bocor)

In [ ]:
std = df[df.label == 1]
subjects = np.array(sorted(std.subject.unique()))
rng = np.random.default_rng(SEED); rng.shuffle(subjects)
train_subj = set(subjects[:int(round(0.8 * len(subjects)))])
train = std[std.subject.isin(train_subj)]
val_std = std[~std.subject.isin(train_subj)]
print('subjek train:', len(train_subj), '| subjek validasi:', len(subjects) - len(train_subj))

## 5. Fitting regresi + toleransi

In [ ]:
k, b = np.polyfit(train.knee, train.hip, 1)
sigma = float(np.std(train.hip - (k * train.knee + b)))
tol = float(min(3 * sigma, 15))

def lowest(clip):  # frame dengan panggul paling bawah (y gambar membesar ke bawah)
    return clip.loc[clip.hy.idxmax()]

gaps = np.array([(r.ky - r.hy) / r.thigh for r in (lowest(c) for _, c in train.groupby('clip'))])
delta = float(max(0.0, np.percentile(gaps, 95)))

ratios = np.array([(np.abs(c.hip - (k * c.knee + b)) <= tol).mean() for _, c in train.groupby('clip')])
min_pass = float(min(0.9, np.percentile(ratios, 5)))

print(f'k={k:.4f} b={b:.4f} (paper: k=0.9525 b=4.1593)')
print(f'sigma={sigma:.2f}  tol={tol:.2f}  delta={delta:.3f}  minPassRatio={min_pass:.2f}')

## 6. Validasi, logika sama dengan `evaluateRep()` di web

In [ ]:
def is_standard(c):
    coord_ok = (np.abs(c.hip - (k * c.knee + b)) <= tol).mean() >= min_pass
    r = lowest(c)
    depth_ok = r.hy >= r.ky - delta * r.thigh
    return bool(coord_ok and depth_ok)

val = pd.concat([val_std, df[df.label == 0]])
res = pd.DataFrame([(lab, is_standard(c)) for (clip, lab), c in val.groupby(['clip', 'label'])], columns=['label', 'pred'])
res['correct'] = res.pred == (res.label == 1)
print('akurasi keseluruhan  :', f'{res.correct.mean():.2%}', '(paper 91.67%)')
print('akurasi squat standar:', f'{res[res.label==1].correct.mean():.2%}', '(paper 98.61%)')
print('akurasi tidak standar:', f'{res[res.label==0].correct.mean():.2%}')
pd.crosstab(res.label, res.pred, rownames=['label (1=standar)'], colnames=['prediksi standar'])

## 7. Simpan & download `waras_params.json`

In [ ]:
params = {'source': 'colab/waras_fit.ipynb on Zenodo data', 'k': round(float(k), 4), 'b': round(float(b), 4),
          'tol': round(tol, 2), 'delta': round(delta, 3), 'minPassRatio': round(min_pass, 2)}
json.dump(params, open('waras_params.json', 'w'), indent=2)
print(params)
files.download('waras_params.json')